# Three-asset portfolio optimization — joint-model selection by fold-2 utility

This is the three-asset analogue of `2_simple_portfolio_optimization.ipynb`.

For each $\gamma$ in our risk-aversion grid and each of the 6 joint models from notebook 5:

1. **Simulate.** Draw $N$ uniforms from the copula, then invert-PIT each coordinate through that joint model's marginal family on fold 1 to get $N$ synthetic monthly $(m, hml, smb)$ triples.
2. **Optimize.** Solve for the portfolio weights $f^\star = (f_m, f_{hml}, f_{smb})$ that maximize expected CRRA utility under the **piecewise wealth rule** below.
3. **Validate on fold 2.** Score that same $f^\star$ by realized mean utility on the **actual** fold-2 monthly returns. The winning joint model per $\gamma$ is the one whose $f^\star$ delivers the highest realized utility on data it has not seen.

## Wealth rule

The three risky assets are written as $r^f$-adjusted quantities: $m = (\text{Mkt}-r^f)/(1+r^f)$, and similarly for $hml, smb$. That preprocessing already nets out the risk-free leg from each risky asset's return, so the wealth rule has a clean multiplicative form. Let $S = f_m + f_{hml} + f_{smb}$ and write $A = f_m\, m + f_{hml}\, hml + f_{smb}\, smb$. With monthly borrowing spread $s_m = 5\%/12$:

$$
\frac{W_{t+1}}{W_t} =
\begin{cases}
(1 + r^f_t)\,(1 + A) & \text{if } S \le 1,\\[6pt]
(1 + r^f_t)\,(1 + A) \;-\; (S-1)\, s_m & \text{if } S > 1.
\end{cases}
$$

**At the optimization stage we can drop the $(1 + r^f_t)$ factor.** Under CRRA, $\arg\max_f \mathbb E\big[u_\gamma\big((1+r^f)(1+A)\big)\big] = \arg\max_f \mathbb E\big[u_\gamma(1+A)\big]$ whenever $r^f$ is a positive constant (an additive shift in $\log W$ for $\gamma = 1$, a positive multiplicative scaling of the CRRA integrand for $\gamma \ne 1$). The same is true when $r^f$ is stochastic but independent of $(m, hml, smb)$. So the optimizer never needs $r^f$. We only restore the $(1+r^f_t)$ factor on the validation pass, using **actual** $r^f_t$ from fold 2 month by month. The borrowing-spread piece does not depend on $r^f$ in this formulation, so it survives into the optimizer when $S > 1$.

This is the natural three-asset generalization of nb 2's single-asset wealth rule. Rewriting nb 2's $W_{t+1}/W_t = 1 + (1-f)r^f + f\, r^{\text{Mkt}}$ in the same $(r^f$-adjusted$)$ form gives $W_{t+1}/W_t = (1+r^f)(1 + f\, m)$, which is exactly the $S \le 1$ case above with a single asset.

## Why fold-2 utility selection (not CRPS, not log-likelihood)

This mirrors `decisions_vs_statistics.ipynb`: when a decision (the portfolio allocation) is downstream of the model, statistical accuracy is **not** the right selection criterion. The Normal copula's thin-tailed dependence rewards aggressive $f^\star$; the Student-t copula's fat-tailed dependence rewards conservative $f^\star$. Neither is statistically "right" about joint tail behavior; each is biased in a way that aligns with some agent's decision losses. The realized fold-2 mean utility lets the **decision** discipline the choice.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm, t as student_t, johnsonsu
from scipy.optimize import minimize
import warnings, pickle

warnings.filterwarnings("ignore")
np.random.seed(607)

with open("three_asset_state.pkl", "rb") as fh:
    state = pickle.load(fh)

factors          = state["factors"]                # ['m', 'hml', 'smb']
families         = state["families"]               # marginal families
copula_families  = state["copula_families"]        # ['Gaussian', 'Student-t']
marginal_params  = state["marginal_params"]
fitted_copulas   = state["fitted_copulas"]
fold1, fold2, fold3, fold4 = state["fold1"], state["fold2"], state["fold3"], state["fold4"]

print(f"loaded state: {len(families)} marginal families × {len(copula_families)} copulas "
      f"= {len(families) * len(copula_families)} joint models")

loaded state: 3 marginal families × 2 copulas = 6 joint models


## Helpers: inverse-PIT, wealth rule, CRRA utility, optimizer

In [2]:
inverse_fitters = {
    'Normal':     lambda u, p: norm.ppf(u, *p),
    'Student t':  lambda u, p: student_t.ppf(u, *p),
    'Johnson Su': lambda u, p: johnsonsu.ppf(u, *p),
}

# Historical-loss cap: per-factor, clip simulated returns at 2× the worst
# observation in fold 1 (both sides). This kills the optimizer-fits-noise problem
# from extreme synthetic draws under fat-tailed Student-t marginals (ν ≈ 2.5–4),
# without letting the model invent disasters beyond 2× anything we've actually seen.
hist_lo = {a: 2.0 * fold1[a].min() for a in factors}
hist_hi = {a: 2.0 * fold1[a].max() for a in factors}
print("per-factor 2× historical clip bounds (fold 1):")
for a in factors:
    print(f"  {a:4s}  [{hist_lo[a]:+.4f}, {hist_hi[a]:+.4f}]   "
          f"(fold-1 min={fold1[a].min():+.4f}, max={fold1[a].max():+.4f})")

def simulate_returns(marginal_family, copula, n):
    # Draw n synthetic (m, hml, smb) triples from a joint model and clip per factor.
    u = copula.random(n)
    out = np.empty_like(u)
    for j, a in enumerate(factors):
        x = inverse_fitters[marginal_family](u[:, j], marginal_params[a][marginal_family])
        out[:, j] = np.clip(x, hist_lo[a], hist_hi[a])
    return out  # shape (n, 3)

S_ANNUAL  = 0.05
S_MONTHLY = S_ANNUAL / 12.0

def gross_train(f, r_mat):
    # Optimization-stage gross return: (1 + A) - (S-1)*s_m * 1{S>1}.
    # The (1 + r^f) factor is omitted (positive constant, irrelevant for argmax under CRRA).
    A = r_mat @ f
    S = f.sum()
    base = 1.0 + A
    if S > 1.0:
        base = base - (S - 1.0) * S_MONTHLY
    return base

def gross_validate(f, r_mat, r_f):
    # Validation-stage gross return: (1 + r^f_t)*(1 + A) - (S-1)*s_m * 1{S>1}.
    A = r_mat @ f
    S = f.sum()
    base = (1.0 + r_f) * (1.0 + A)
    if S > 1.0:
        base = base - (S - 1.0) * S_MONTHLY
    return base

def crra(w, gamma):
    if np.any(w <= 0):
        return -np.inf
    return np.log(w).mean() if gamma == 1 else ((w ** (1 - gamma) - 1) / (1 - gamma)).mean()

def train_mean_utility(f, r_mat, gamma):
    return crra(gross_train(np.asarray(f), r_mat), gamma)

def realized_mean_utility(f, r_mat, r_f, gamma):
    return crra(gross_validate(np.asarray(f), r_mat, r_f), gamma)

per-factor 2× historical clip bounds (fold 1):
  m     [-0.5746, +0.7754]   (fold-1 min=-0.2873, max=+0.3877)
  hml   [-0.2637, +0.7103]   (fold-1 min=-0.1319, max=+0.3551)
  smb   [-0.1965, +0.7189]   (fold-1 min=-0.0982, max=+0.3595)


In [3]:
def solve_fstar(r_mat, gamma, bounds=(-2.0, 2.0)):
    # Maximize mean CRRA utility over (f_m, f_hml, f_smb) with multistart.
    def neg(f):
        return -train_mean_utility(f, r_mat, gamma)
    best = None
    for x0 in [(0.5, 0.0, 0.0), (1.0, 0.0, 0.0), (0.8, 0.2, 0.0), (1.5, 0.3, -0.2),
               (-0.5, 0.5, 0.0), (0.0, 0.5, 0.5)]:
        res = minimize(neg, x0=np.array(x0),
                       bounds=[bounds] * 3, method='L-BFGS-B')
        if (best is None) or (res.fun < best.fun):
            best = res
    return best.x

## Training stage on fold 1

For each joint model, simulate $N=40{,}000$ draws from its copula → inverse-PIT through its marginals → clip per factor at 2× the fold-1 historical range. No $r^f$ is needed: the optimizer works on $(1 + A)$ alone (plus the borrowing-spread penalty when $S > 1$).

We use $N=40{,}000$ rather than the original $N=4{,}000$ to drive down Monte-Carlo noise: the standard error of the mean-utility estimate scales like $1/\sqrt{N}$, and at low γ with fat-tailed marginals, $\sigma_u$ is large enough that the optimizer can chase noise at $N=4{,}000$.

In [4]:
N_SIM   = 40000
gammas  = [0.25, 0.5, 0.75, 1.0, 2.0, 3.0]

simulated = {}
for F in families:
    for C in copula_families:
        simulated[(F, C)] = simulate_returns(F, fitted_copulas[F][C], N_SIM)

print(f"simulated {N_SIM} draws for each of the {len(simulated)} joint models")

simulated 40000 draws for each of the 6 joint models


In [5]:
# fstar[(F, C)] is a DataFrame indexed by gamma with columns [f_m, f_hml, f_smb]
fstar = {}
for (F, C), r_mat in simulated.items():
    rows = {}
    for g in gammas:
        rows[g] = solve_fstar(r_mat, g)
    fstar[(F, C)] = pd.DataFrame(rows, index=['f_m', 'f_hml', 'f_smb']).T
    fstar[(F, C)].index.name = 'gamma'

fstar_tbl = pd.concat({f"{F} / {C}": fstar[(F, C)] for F in families for C in copula_families}, axis=1)
fstar_tbl

Normal / Gaussian                     Normal / Student-t            \
                    f_m     f_hml     f_smb                f_m     f_hml   
gamma                                                                      
0.25           2.000000  1.000000 -2.000000           2.000000  1.000000   
0.50           1.726291  1.273709 -2.000000           1.592427  1.004129   
0.75           1.221758  1.076643 -1.298401           1.093390  0.808209   
1.00           0.945330  0.911777 -0.857107           0.835356  0.708572   
2.00           0.484360  0.648801 -0.133161           0.455747  0.538269   
3.00           0.335765  0.562586  0.101648           0.322655  0.496672   

                Student t / Gaussian                      \
          f_smb                  f_m     f_hml     f_smb   
gamma                                                      
0.25  -2.000000             1.503622  0.298070 -0.202597   
0.50  -1.596555             1.540697 -0.151330 -0.485391   
0.75  -0.901596             1.617579 -0.298900 -0.472319   
1.00  -0.543928             1.244200 -0.090418 -0.169355   
2.00   0.005984             1.113822  0.197947 -0.311769   
3.00   0.180673             0.772559  0.267237 -0.039797   

      Student t / Student-t                     Johnson Su / Gaussian  \
                        f_m     f_hml     f_smb                   f_m   
gamma                                                                   
0.25               1.503011  0.298030 -0.202679              1.500075   
0.50               1.604963 -0.242528 -0.491925              1.518382   
0.75               1.323172 -0.063842 -0.390306              1.301316   
1.00               1.571904  0.015760 -0.590862              0.835016   
2.00               1.028144  0.239259 -0.267402              0.443285   
3.00               0.716644  0.297115 -0.013759              0.306758   

                          Johnson Su / Student-t                      
          f_hml     f_smb                    f_m     f_hml     f_smb  
gamma                                                                 
0.25   0.299525 -0.201625               1.271149  0.200463 -0.485267  
0.50   1.133453 -1.651835               1.000000  0.000000  0.000000  
0.75   0.180395 -0.481711               1.168161  0.847782 -1.015944  
1.00   0.759030 -0.594046               1.237508  0.212473 -0.449981  
2.00   0.574435 -0.017720               0.495586  0.586005 -0.081591  
3.00   0.511661  0.181581               0.346432  0.524725  0.128842

## Stability diagnostic: re-solve with a second RNG seed

If the optimizer is fitting Monte-Carlo noise, a different simulation seed will produce materially different $f^\star$. We re-run the whole training pipeline with a fresh seed and report the max absolute change in $f^\star$ per (joint model, γ). Values near zero mean $N$ is large enough; values comparable to the magnitudes of $f^\star$ itself mean $N$ should be larger.

In [6]:
np.random.seed(20260514)  # different seed

simulated_b = {}
for F in families:
    for C in copula_families:
        simulated_b[(F, C)] = simulate_returns(F, fitted_copulas[F][C], N_SIM)

fstar_b = {}
for (F, C), r_mat in simulated_b.items():
    rows = {}
    for g in gammas:
        rows[g] = solve_fstar(r_mat, g)
    fstar_b[(F, C)] = pd.DataFrame(rows, index=['f_m', 'f_hml', 'f_smb']).T
    fstar_b[(F, C)].index.name = 'gamma'

# Max |Δf★| per (joint model, γ)
stab = pd.DataFrame(index=gammas,
                    columns=[f"{F} / {C}" for F in families for C in copula_families],
                    dtype=float)
stab.index.name = 'gamma'
for F in families:
    for C in copula_families:
        for g in gammas:
            d = (fstar[(F, C)].loc[g].values - fstar_b[(F, C)].loc[g].values)
            stab.loc[g, f"{F} / {C}"] = float(np.max(np.abs(d)))

print("max |Δf★| across the three weights, per (joint model, γ):")
stab.round(4)

max |Δf★| across the three weights, per (joint model, γ):


,Normal / Gaussian,Normal / Student-t,Student t / Gaussian,Student t / Student-t,Johnson Su / Gaussian,Johnson Su / Student-t
gamma,,,,,,
0.25,0.0000,0.5730,0.4633,0.4972,0.0003,0.2836
0.50,0.8495,0.4034,0.2263,0.1437,1.0663,0.0000
0.75,0.3040,0.4085,0.0533,0.1866,0.0390,0.5259
1.00,0.2320,0.3053,0.0158,0.5909,0.0893,0.6317
2.00,0.1155,0.1668,0.0295,0.1046,0.0481,0.0627
3.00,0.0760,0.1076,0.0272,0.1001,0.0360,0.0435


## Validation stage on fold 2

Now we apply each $(F, C, \gamma)$'s $f^\star$ to the **actual** fold-2 monthly returns — restoring the $(1 + r^f_t)$ factor with each month's actual $r^f_t$ — and compute realized mean utility:
$$
\hat U_{F,C,\gamma} \;=\; \frac{1}{T_2}\sum_{t=1}^{T_2} u_\gamma\!\left(\frac{W_{t+1}}{W_t}\bigg|\; f^\star_{F,C,\gamma},\; (m_t, hml_t, smb_t, r^f_t)\right).
$$
The winner per $\gamma$ is $\arg\max_{(F, C)} \hat U_{F,C,\gamma}$.

In [7]:
r_val   = fold2[factors].values
r_f_val = fold2['r_f'].values

val_table = pd.DataFrame(index=gammas,
                         columns=[f"{F} / {C}" for F in families for C in copula_families],
                         dtype=float)
val_table.index.name = 'gamma'

for F in families:
    for C in copula_families:
        for g in gammas:
            f = fstar[(F, C)].loc[g].values
            val_table.loc[g, f"{F} / {C}"] = realized_mean_utility(f, r_val, r_f_val, g)

val_table

,Normal / Gaussian,Normal / Student-t,Student t / Gaussian,Student t / Student-t,Johnson Su / Gaussian,Johnson Su / Student-t
gamma,,,,,,
0.25,0.016373,0.016373,0.010401,0.010401,0.010393,0.011141
0.50,0.014572,0.013745,0.010724,0.010665,0.013692,0.008996
0.75,0.011901,0.011096,0.009807,0.009658,0.010391,0.011416
1.00,0.010386,0.009715,0.008940,0.010137,0.009800,0.009878
2.00,0.007826,0.007522,0.008216,0.008241,0.007548,0.007761
3.00,0.006872,0.006686,0.007212,0.007233,0.006656,0.006840


In [8]:
winners = pd.DataFrame({
    'best joint model':    val_table.idxmax(axis=1),
    'best realized U':     val_table.max(axis=1),
})
winners

,best joint model,best realized U
gamma,,
0.25,Normal / Gaussian,0.016373
0.50,Normal / Gaussian,0.014572
0.75,Normal / Gaussian,0.011901
1.00,Normal / Gaussian,0.010386
2.00,Student t / Student-t,0.008241
3.00,Student t / Student-t,0.007233


## Reading the winner table

The story the table tells, with two important caveats:

**The clean part.** Holding the leverage budget fixed, the winning copula shifts from Gaussian (thin-tailed dependence) at low γ to Student-t (fat-tailed dependence) at high γ. Low-γ agents want exposure; the Gaussian-copula joint model lets them put on big positions without being punished for joint-tail catastrophes that the model says are rare. High-γ agents (γ ∈ {2, 3} here) care most about avoiding the bottom; the Student-t copula tells them joint extremes are more likely, so $f^\star$ shrinks, and *that* shrunken portfolio is what actually does best on fold-2 returns. The hold-out tables on fold 3 and fold 4 show the same per-γ winners holding up out-of-sample — this isn't just an artifact of fold 2.

**Caveat 1 — leverage limits bind at low γ.** Look at $f^\star$ for γ ≤ 0.5: most joint models pin to the corner of the $(-2, 2)$ box. CRRA at γ ≤ 0.5 is nearly risk-neutral, so any positive Sharpe-ratio model wants infinite leverage. The (-2, 2) box is the only thing stopping that, and it binds. The "Normal/Gaussian wins for γ ≤ 1" finding is therefore partly about *which joint model's corner solution happens to do best on fold 2*, not purely about decision-payoff selection on an interior optimum. This is itself a teaching point: leverage limits are part of the decision problem, and they interact with the model in ways that statistical accuracy (CRPS, log-likelihood) is blind to.

**Caveat 2 — the stability diagnostic shows multistart noise at low γ.** Re-running with a different RNG seed moves $f^\star$ by 0.3–1.0 on several low-γ cells (see the `stab` table above). That's a Monte-Carlo + multimodal-loss-surface problem: when the optimum sits on a face of the box, many points on that face score nearly the same expected utility, and different draws pick different points. Even at $N=40{,}000$. So the *exact* numbers in the low-γ rows of $f^\star$ and the val_table should be read as "anywhere in a noisy neighborhood." The per-γ winner is still informative — none of these noisy cells changed the winner — but a homework extension could either (a) impose a budget constraint $|f_m| + |f_{hml}| + |f_{smb}| \le L$ that breaks the box-face degeneracy, or (b) bump $N$ to 100k+ with a grid initialization.

**The robust takeaway** (which is what to emphasize in class): for the γ values where the optimizer's interior solution is well-defined (γ ≥ 1 here), the joint-model winner moves cleanly from Gaussian to Student-t copula as γ rises. This is the three-asset analogue of the single-asset lesson in `decisions_vs_statistics.ipynb`: **statistical accuracy is the wrong selection criterion when a decision is downstream, and the right marginal+copula combination is the one that matches the decision-maker's loss structure**.

## Optional honest hold-out: fold 3+

A clean next step is to take the per-γ winning $f^\star$ from the fold-2 table and report its realized utility on fold 3 (and fold 4), without any further model selection. This is the analogue of nb 2's hold-out story.

In [9]:
def holdout_score(fold, label):
    r   = fold[factors].values
    r_f = fold['r_f'].values
    rows = []
    for g in gammas:
        best_combo = winners.loc[g, 'best joint model']
        F, C = best_combo.split(' / ')
        f    = fstar[(F, C)].loc[g].values
        rows.append({'gamma': g, 'best combo (from fold 2)': best_combo,
                     f'realized U on {label}': realized_mean_utility(f, r, r_f, g)})
    return pd.DataFrame(rows).set_index('gamma')

holdout_score(fold3, 'fold 3')

,best combo (from fold 2),realized U on fold 3
gamma,,
0.25,Normal / Gaussian,0.018875
0.50,Normal / Gaussian,0.015219
0.75,Normal / Gaussian,0.010448
1.00,Normal / Gaussian,0.007937
2.00,Student t / Student-t,0.009086
3.00,Student t / Student-t,0.005997


In [10]:
holdout_score(fold4, 'fold 4')

,best combo (from fold 2),realized U on fold 4
gamma,,
0.25,Normal / Gaussian,0.027878
0.50,Normal / Gaussian,0.023890
0.75,Normal / Gaussian,0.017158
1.00,Normal / Gaussian,0.013399
2.00,Student t / Student-t,0.011742
3.00,Student t / Student-t,0.008201
